In [ ]:
from config_reader import parseCommonCfg, parsePeerInfo
import subprocess
import sys
import time
import glob

common_cfg = parseCommonCfg('Common.cfg')
peers = parsePeerInfo('PeerInfo.cfg')

for key, value in common_cfg.items():
    print(f"{key}: {value}")

for p in peers:
    status = "has file" if p['has_file'] else "no file"
    print(f"Peer {p['id']} @ {p['host']}:{p['port']} ({status})")

In [ ]:
processes = {}

for peer in peers:
    pid = peer['id']
    print(f"Launching Peer {pid}...")
    proc = subprocess.Popen(
        [sys.executable, 'peerProcess.py', str(pid)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    processes[pid] = proc
    time.sleep(1.5)

print(f"All {len(processes)} peers launched.")

In [ ]:
for log_file in sorted(glob.glob('log_peer_*.log')):
    print(f"\n{'='*60}")
    print(f"  {log_file}")
    print(f"{'='*60}")
    with open(log_file, 'r') as f:
        print(f.read().strip())

In [ ]:
import socket, struct, threading, time, os
from config_reader import parseCommonCfg

MSG_CHOKE=0; MSG_UNCHOKE=1; MSG_INTERESTED=2; MSG_NOT_INTERESTED=3
MSG_HAVE=4;  MSG_BITFIELD=5; MSG_REQUEST=6;   MSG_PIECE=7

HANDSHAKE_HEADER = b'P2PFILESHARINGPROJ'
TEST_PORT   = 19999
SENDER_ID   = 1001
RECEIVER_ID = 1002
PIECE_INDEX = 0
NUM_PIECES  = 3
results     = {}

def make_msg(msg_type, payload=b''):
    return struct.pack('>IB', len(payload)+1, msg_type) + payload

def recv_exact(s, n):
    buf = b''
    while len(buf) < n:
        chunk = s.recv(n - len(buf))
        if not chunk: return None
        buf += chunk
    return buf

def recv_msg(s):
    hdr = recv_exact(s, 4)
    if not hdr: return None, None
    p = recv_exact(s, struct.unpack('>I', hdr)[0])
    if p is None: return None, None
    return p[0], p[1:]

def make_handshake(peer_id):
    return struct.pack('>18s10xI', HANDSHAKE_HEADER, peer_id)

def parse_handshake(data):
    if len(data) != 32: return None
    h, pid = struct.unpack('>18s10xI', data)
    return pid if h == HANDSHAKE_HEADER else None

def make_bitfield_bytes(bits):
    out = bytearray()
    for i in range(0, len(bits), 8):
        b = 0
        for j in range(8):
            if i+j < len(bits) and bits[i+j]: b |= (1 << (7-j))
        out.append(b)
    return bytes(out)

def parse_bitfield_bytes(raw, num_pieces):
    bits = []
    for byte in raw:
        for j in range(7, -1, -1): bits.append((byte >> j) & 1)
    return bits[:num_pieces]

def read_piece(peer_id, piece_index, piece_size, file_size):
    path = os.path.join(f'peer_{peer_id}', 'thefile')
    offset = piece_index * piece_size
    length = min(piece_size, file_size - offset)
    with open(path, 'rb') as f:
        f.seek(offset)
        return f.read(length)

def save_piece(peer_id, piece_index, piece_size, data):
    folder = f'peer_{peer_id}'
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, 'thefile')
    mode = 'r+b' if os.path.exists(path) else 'wb'
    with open(path, mode) as f:
        f.seek(piece_index * piece_size)
        f.write(data)
    print(f'[Receiver] Saved piece {piece_index} to {path}')

def sender_thread(piece_data):
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind(('127.0.0.1', TEST_PORT))
    srv.listen(1)
    results['sender_ready'] = True
    conn, _ = srv.accept()
    try:
        remote_id = parse_handshake(recv_exact(conn, 32))
        assert remote_id == RECEIVER_ID
        conn.send(make_handshake(SENDER_ID))
        print(f'[Sender] Handshake OK with peer {remote_id}')
        conn.send(make_msg(MSG_BITFIELD, make_bitfield_bytes([1]*NUM_PIECES)))
        print('[Sender] Sent bitfield')
        t, p = recv_msg(conn)
        assert t == MSG_BITFIELD
        print(f'[Sender] Got receiver bitfield: {parse_bitfield_bytes(p, NUM_PIECES)}')
        t, _ = recv_msg(conn)
        assert t == MSG_INTERESTED
        print('[Sender] Receiver is interested')
        conn.send(make_msg(MSG_UNCHOKE))
        print('[Sender] Sent UNCHOKE')
        t, p = recv_msg(conn)
        assert t == MSG_REQUEST
        idx = struct.unpack('>I', p)[0]
        print(f'[Sender] Got request for piece {idx}')
        conn.send(make_msg(MSG_PIECE, struct.pack('>I', idx) + piece_data))
        print(f'[Sender] Sent piece {idx} ({len(piece_data)} bytes)')
        results['sender_passed'] = True
    except AssertionError as e:
        results['sender_error'] = str(e)
    finally:
        conn.close(); srv.close()

def receiver_thread(piece_size):
    while not results.get('sender_ready'): time.sleep(0.05)
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.connect(('127.0.0.1', TEST_PORT))
    try:
        sock.send(make_handshake(RECEIVER_ID))
        remote_id = parse_handshake(recv_exact(sock, 32))
        assert remote_id == SENDER_ID
        print(f'[Receiver] Handshake OK with peer {remote_id}')
        t, p = recv_msg(sock)
        assert t == MSG_BITFIELD
        print(f'[Receiver] Got sender bitfield: {parse_bitfield_bytes(p, NUM_PIECES)}')
        sock.send(make_msg(MSG_BITFIELD, make_bitfield_bytes([0]*NUM_PIECES)))
        print('[Receiver] Sent own bitfield')
        sock.send(make_msg(MSG_INTERESTED))
        print('[Receiver] Sent INTERESTED')
        t, _ = recv_msg(sock)
        assert t == MSG_UNCHOKE
        print('[Receiver] Got UNCHOKE')
        sock.send(make_msg(MSG_REQUEST, struct.pack('>I', PIECE_INDEX)))
        print(f'[Receiver] Sent REQUEST for piece {PIECE_INDEX}')
        t, p = recv_msg(sock)
        assert t == MSG_PIECE
        idx = struct.unpack('>I', p[:4])[0]
        received_data = p[4:]
        print(f'[Receiver] Got piece {idx} ({len(received_data)} bytes)')
        save_piece(RECEIVER_ID, idx, piece_size, received_data)
        results['receiver_passed'] = True
        results['received_data']   = received_data
    except AssertionError as e:
        results['receiver_error'] = str(e)
    finally:
        sock.close()

common_cfg = parseCommonCfg('Common.cfg')
piece_size = common_cfg['PieceSize']
file_size  = common_cfg['FileSize']
piece_data = read_piece(SENDER_ID, PIECE_INDEX, piece_size, file_size)

print('=' * 60)
print('  TEST: File piece request and transfer between peers')
print('=' * 60)
s = threading.Thread(target=sender_thread,   args=(piece_data,), daemon=True)
r = threading.Thread(target=receiver_thread, args=(len(piece_data),), daemon=True)
s.start(); r.start()
s.join(timeout=10); r.join(timeout=10)

print('\n--- Results ---')
if results.get('sender_error'):   print(f'[FAIL] Sender: {results["sender_error"]}')
if results.get('receiver_error'): print(f'[FAIL] Receiver: {results["receiver_error"]}')
if results.get('sender_passed') and results.get('receiver_passed'):
    got = results['received_data']
    if got == piece_data:
        print('[ OK ] Piece data transferred correctly!')
        print(f'[ OK ] {len(got)} bytes received, content matches original.')
        print(f'[ OK ] File saved to peer_{RECEIVER_ID}/thefile')
        print('\n✅  PASS — File pieces are successfully requested and sent between peers.')
    else:
        print(f'[FAIL] Data mismatch! Expected {len(piece_data)} bytes, got {len(got)}')
        print('\n❌  FAIL')
else:
    print('\n❌  FAIL — One or both peers did not complete the exchange.')

In [ ]:
for pid, proc in processes.items():
    status = "RUNNING" if proc.poll() is None else f"EXITED (code {proc.returncode})"
    print(f"Peer {pid}: {status}")

In [ ]:
for pid, proc in processes.items():
    if proc.poll() is None:
        proc.terminate()
        proc.wait(timeout=5)
        print(f"Peer {pid}: terminated")
    else:
        print(f"Peer {pid}: already exited (code {proc.returncode})")